# 虎牙回放下载 + DeepFilter 降噪 + 保存到 Drive

这个 notebook 只保留配置和启动逻辑。真正的运行代码放在 GitHub 的 `github_runtime/huya_replay_df_runtime.py`，Colab 运行时会先下载最新版本再执行。

In [ ]:
%pip -q install -U yt-dlp
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

#@title 1. GitHub 运行时代码
GITHUB_RAW_BASE_URL = 'https://raw.githubusercontent.com/yourname/yourrepo/main/github_runtime' #@param {type:"string"}
RUNTIME_FILENAME = 'huya_replay_df_runtime.py' #@param {type:"string"}
RUNTIME_MODULE_NAME = 'huya_replay_df_runtime' #@param {type:"string"}
FORCE_RUNTIME_DOWNLOAD = True #@param {type:"boolean"}

#@title 2. 下载与降噪参数
HUYA_URL = 'https://www.huya.com/video/play/1111005562.html' #@param {type:"string"}
DRIVE_ROOT = '/content/drive/MyDrive/huya_replay_df' #@param {type:"string"}
JOB_NAME = '' #@param {type:"string"}
USE_DRIVE_CACHED_RAW_VIDEO = False #@param {type:"boolean"}
CACHED_RAW_VIDEO_PATH = '' #@param {type:"string"}
COPY_RAW_TO_DRIVE_IMMEDIATELY = True #@param {type:"boolean"}
SEGMENT_DURATION_MINUTES = 8 #@param {type:"integer"}
YTDLP_CONCURRENT_FRAGMENTS = 8 #@param {type:"integer"}
DF_MAX_WORKERS = 1 #@param {type:"integer"}
ENABLE_POSTFILTER = False #@param {type:"boolean"}
KEEP_WORKFILES = False #@param {type:"boolean"}

#@title 3. 音频与 DeepFilter 参数
AUDIO_SAMPLE_RATE = 48000 #@param [16000, 24000, 44100, 48000] {type:"raw"}
DEEP_FILTER_BINARY = 'deep-filter-0.5.6-x86_64-unknown-linux-musl' #@param {type:"string"}
WORK_ROOT = '/content/huya_df_work' #@param {type:"string"}


In [ ]:
import importlib.util
import sys
import time
import urllib.request
from pathlib import Path

runtime_dir = Path('/content/codex_runtime')
runtime_dir.mkdir(parents=True, exist_ok=True)
if 'yourname/yourrepo' in GITHUB_RAW_BASE_URL:
    raise ValueError('请先把 GITHUB_RAW_BASE_URL 改成你自己的 GitHub raw 目录。')
runtime_url = f"{GITHUB_RAW_BASE_URL.rstrip('/')}/{RUNTIME_FILENAME}"
download_url = runtime_url
if FORCE_RUNTIME_DOWNLOAD:
    download_url = f"{runtime_url}?ts={int(time.time())}"

local_runtime_path = runtime_dir / RUNTIME_FILENAME
print('Downloading runtime from:', download_url)
urllib.request.urlretrieve(download_url, local_runtime_path)
print('Runtime saved to:', local_runtime_path)

spec = importlib.util.spec_from_file_location(RUNTIME_MODULE_NAME, local_runtime_path)
runtime_module = importlib.util.module_from_spec(spec)
sys.modules[RUNTIME_MODULE_NAME] = runtime_module
spec.loader.exec_module(runtime_module)
print('Runtime module loaded:', runtime_module)


In [ ]:
config = {
    'huya_url': HUYA_URL,
    'drive_root': DRIVE_ROOT,
    'job_name': JOB_NAME,
    'use_drive_cached_raw_video': USE_DRIVE_CACHED_RAW_VIDEO,
    'cached_raw_video_path': CACHED_RAW_VIDEO_PATH,
    'copy_raw_to_drive_immediately': COPY_RAW_TO_DRIVE_IMMEDIATELY,
    'segment_duration_minutes': SEGMENT_DURATION_MINUTES,
    'ytdlp_concurrent_fragments': YTDLP_CONCURRENT_FRAGMENTS,
    'df_max_workers': DF_MAX_WORKERS,
    'enable_postfilter': ENABLE_POSTFILTER,
    'keep_workfiles': KEEP_WORKFILES,
    'audio_sample_rate': AUDIO_SAMPLE_RATE,
    'deep_filter_binary': DEEP_FILTER_BINARY,
    'work_root': WORK_ROOT,
}

result = runtime_module.run_pipeline(config)
result
